## Day 3: Baselines and Linear Regression

This notebook implements:
1) baseline volatility forecasts,
2) linear regression forecasting,
3) MAE/RMSE evaluation,
4) prediction-vs-realized plots for AAPL and MSFT.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name.startswith("notebooks") else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data:"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data"

FIG_DIR = PROJECT_ROOT / "figures:"
if not FIG_DIR.exists():
    FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Using data dir:", DATA_DIR)
print("Using figures dir:", FIG_DIR)

In [ ]:
def add_volatility(df, window=10):
    df = df.sort_values("Date").copy()
    df["log_return"] = np.log(df["Close"]).diff()
    df["rv_10"] = df["log_return"].rolling(window).std()
    return df


def make_target(df):
    df = df.sort_values("Date").copy()
    df["target_rv_10"] = df["rv_10"].shift(-1)
    return df


def add_features(df, n_return_lags=5):
    df = df.sort_values("Date").copy()

    df["log_close"] = np.log(df["Close"])

    for k in range(1, n_return_lags + 1):
        df[f"log_return_lag_{k}"] = df["log_return"].shift(k)

    df["rv_10_lag_1"] = df["rv_10"].shift(1)
    df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))
    df["volume_change"] = df["Volume"].pct_change()

    return df


def build_day2_ready_df(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    # Convert to tz-naive timestamps for consistent time slicing.
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)

    df = make_target(add_volatility(df, window=10))
    df = add_features(df, n_return_lags=5)
    df = df.dropna().sort_values("Date").reset_index(drop=True)
    return df


aapl = build_day2_ready_df(DATA_DIR / "AAPL_clean.csv")
msft = build_day2_ready_df(DATA_DIR / "MSFT_clean.csv")

print("AAPL rows:", len(aapl), "MSFT rows:", len(msft))
aapl.head()

In [ ]:
feature_cols = [
    c
    for c in aapl.columns
    if c.startswith("log_return_lag_")
    or c in ["log_close", "rv_10", "rv_10_lag_1", "log_volume", "volume_change"]
]
target_col = "target_rv_10"


def time_split(df, train_end, val_end):
    train = df[df["Date"] <= train_end].copy()
    val = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
    test = df[df["Date"] > val_end].copy()
    return train, val, test


train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2022-12-31")

a_train, a_val, a_test = time_split(aapl, train_end, val_end)
m_train, m_val, m_test = time_split(msft, train_end, val_end)

print("Feature columns:", feature_cols)
print("AAPL split sizes:", len(a_train), len(a_val), len(a_test))
print("MSFT split sizes:", len(m_train), len(m_val), len(m_test))

In [ ]:
def naive_baseline(df):
    df = df.sort_values("Date").copy()
    # target_rv_10 at row t is rv_10 at row t+1, so current rv_10 is a valid naive forecast.
    df["pred_naive"] = df["rv_10"]
    return df


def rolling_mean_baseline(df, window=10):
    df = df.sort_values("Date").copy()
    df["pred_rollmean"] = df["rv_10"].rolling(window).mean()
    return df


def add_baselines(train, val, test):
    combined = pd.concat([train, val, test], axis=0).sort_values("Date").copy()
    combined = naive_baseline(combined)
    combined = rolling_mean_baseline(combined, window=10)
    combined = combined.dropna(subset=["pred_naive", "pred_rollmean", target_col])
    return time_split(combined, train_end, val_end)


def eval_forecasts(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse


def evaluate_baselines(df_test, ticker):
    y_true = df_test[target_col].values
    mae_naive, rmse_naive = eval_forecasts(y_true, df_test["pred_naive"])
    mae_rm, rmse_rm = eval_forecasts(y_true, df_test["pred_rollmean"])
    return [
        {"Ticker": ticker, "Model": "Naive", "Split": "Test", "MAE": mae_naive, "RMSE": rmse_naive},
        {"Ticker": ticker, "Model": "RollingMean10", "Split": "Test", "MAE": mae_rm, "RMSE": rmse_rm},
    ]


a_train, a_val, a_test = add_baselines(a_train, a_val, a_test)
m_train, m_val, m_test = add_baselines(m_train, m_val, m_test)

baseline_rows = []
baseline_rows.extend(evaluate_baselines(a_test, "AAPL"))
baseline_rows.extend(evaluate_baselines(m_test, "MSFT"))

baseline_results = pd.DataFrame(baseline_rows)
baseline_results

In [ ]:
def fit_linear(train, val, test, label):
    X_train = train[feature_cols].values
    y_train = train[target_col].values
    X_val = val[feature_cols].values
    y_val = val[target_col].values
    X_test = test[feature_cols].values
    y_test = test[target_col].values

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    mae_val, rmse_val = eval_forecasts(y_val, y_val_pred)
    mae_test, rmse_test = eval_forecasts(y_test, y_test_pred)

    test_out = test.copy()
    test_out["pred_linear"] = y_test_pred

    metrics_rows = [
        {"Ticker": label, "Model": "LinearRegression", "Split": "Val", "MAE": mae_val, "RMSE": rmse_val},
        {"Ticker": label, "Model": "LinearRegression", "Split": "Test", "MAE": mae_test, "RMSE": rmse_test},
    ]
    return model, test_out, metrics_rows


a_lr, a_test, a_lr_rows = fit_linear(a_train, a_val, a_test, "AAPL")
m_lr, m_test, m_lr_rows = fit_linear(m_train, m_val, m_test, "MSFT")

linear_results = pd.DataFrame(a_lr_rows + m_lr_rows)
linear_results

In [ ]:
results = pd.concat([baseline_results, linear_results], ignore_index=True)
results = results.sort_values(["Ticker", "Split", "RMSE"]).reset_index(drop=True)

results

In [ ]:
RESULTS_DIR = DATA_DIR / "results_day3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(RESULTS_DIR / "metrics_day3.csv", index=False)
a_test.to_csv(RESULTS_DIR / "AAPL_test_with_predictions.csv", index=False)
m_test.to_csv(RESULTS_DIR / "MSFT_test_with_predictions.csv", index=False)

print("Saved result files to:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*.csv")):
    print(" -", p.name)

In [ ]:
def plot_preds(df_test, ticker):
    plt.figure(figsize=(10, 4))
    plt.plot(df_test["Date"], df_test[target_col], label="Realized RV", linewidth=1.8)
    plt.plot(df_test["Date"], df_test["pred_naive"], label="Naive", alpha=0.8)
    plt.plot(df_test["Date"], df_test["pred_linear"], label="Linear", alpha=0.8)
    plt.title(f"{ticker} - 1-day-ahead 10-day Realized Volatility")
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{ticker}_baseline_vs_linear.png", dpi=150)
    plt.close()


plot_preds(a_test, "AAPL")
plot_preds(m_test, "MSFT")

print("Saved plots:")
print(" -", (FIG_DIR / "AAPL_baseline_vs_linear.png").name)
print(" -", (FIG_DIR / "MSFT_baseline_vs_linear.png").name)

### Day 3 Checklist (Implemented)

- Baselines: naive and rolling-mean forecasts implemented and evaluated.
- ML model: linear regression trained on train, validated on val, tested on test.
- Metrics: MAE and RMSE table created and saved.
- Figures: baseline-vs-linear prediction plots saved for AAPL and MSFT.